# Evaluation — YOLOE → Any6D on the RGB-D Object Dataset

**Pipeline:** YOLOE (visual-prompt detection) → Any6D (FoundationPose-based 6D pose estimation)  
**Dataset:** RGB-D Object Dataset · University of Washington · 300 household objects  
**Metric:** ADD (Average Distance of Model Points), AUC@10 cm, detection rate  
**Scope:** 10 representative objects, 10 frames each

```
Anchor frame (ref)
      ↓
   YOLOE (visual prompt)     → bbox + mask
      ↓
   Any6D (FoundationPose)    → 4×4 pose matrix
      ↓
   ADD metric vs ground truth
```

**Run inside the Any6D Docker container:**
```bash
docker compose run --rm -p 8888:8888 any6d jupyter lab --ip=0.0.0.0 --no-browser
```

---
## Cell 1 — Imports & Environment

In [1]:
import os, sys, json, warnings
import numpy as np
import cv2
import trimesh
import torch
import tempfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from PIL import Image

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable

warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':        150,
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'axes.labelpad':     6,
    'xtick.major.size':  4,
    'ytick.major.size':  4,
    'legend.frameon':    False,
    'legend.fontsize':   9,
})

# Palette — matches thesis figures
C_BLUE   = '#2C6FAC'
C_GREEN  = '#2E8B57'
C_RED    = '#C0392B'
C_ORANGE = '#E67E22'
C_GREY   = '#BDC3C7'
C_DARK   = '#2C3E50'

# ── Any6D paths ───────────────────────────────────────────────
WORKSPACE = '/workspace'
sys.path.insert(0, WORKSPACE)
sys.path.insert(0, f'{WORKSPACE}/foundationpose')
sys.path.insert(0, f'{WORKSPACE}/foundationpose/mycpp/build')

import nvdiffrast.torch as dr
from estimater import Any6D
from foundationpose.Utils import get_bounding_box

from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

print('✅ All imports OK')
print(f'   PyTorch  {torch.__version__}')
print(f'   CUDA     {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—"}')

ModuleNotFoundError: No module named 'nvdiffrast'

---
## Cell 2 — Configuration

In [ ]:
# ── Paths — adjust to your setup ─────────────────────────────
DATASET_ROOT  = '/data/rgbd-dataset'          # root of the UW RGB-D dataset
MESH_ROOT     = '/data/meshes'                # one <instance>.obj per object
SAVE_DIR      = Path('/workspace/results/eval_rgbd')
YOLOE_WEIGHTS = 'yoloe-11l-seg.pt'
N_FRAMES      = 10                            # frames evaluated per object
ADD_THRESHOLD = 10.0                          # cm — AUC threshold

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Kinect intrinsics (RGB-D Object Dataset) ──────────────────
K_RGBD = np.array([
    [525.0,   0.0, 319.5],
    [  0.0, 525.0, 239.5],
    [  0.0,   0.0,   1.0],
], dtype=np.float64)

DEPTH_SCALE = 1000.0   # mm → m

# ── 10 evaluation objects ─────────────────────────────────────
# (category, instance, sequence_id)
EVAL_OBJECTS = [
    ('coffee_mug',   'coffee_mug',            1),
    ('cereal_box',   'frosted_mini_wheats',    1),
    ('soda_can',     'pepsi_can',              1),
    ('apple',        'fuji_apple',             1),
    ('stapler',      'stapler',                1),
    ('scissors',     'scissors_2',             1),
    ('flashlight',   'flashlight_2',           1),
    ('kleenex',      'kleenex_tissue_box',     1),
    ('notebook',     'notebook',               1),
    ('pitcher',      'water_pitcher',          1),
]

OBJECT_LABELS = [inst for _, inst, _ in EVAL_OBJECTS]

print(f'Evaluating {len(EVAL_OBJECTS)} objects × {N_FRAMES} frames = {len(EVAL_OBJECTS)*N_FRAMES} total runs')
for cat, inst, seq in EVAL_OBJECTS:
    print(f'  {inst:<30} ({cat}, seq {seq})')

---
## Cell 3 — Data Structures & Metrics

In [ ]:
@dataclass
class FrameResult:
    object_name: str
    frame_id:    int
    add_cm:      float
    detected:    bool
    pred_pose:   np.ndarray
    gt_pose:     np.ndarray


@dataclass
class ObjectResult:
    name:   str
    frames: List[FrameResult] = field(default_factory=list)

    @property
    def detection_rate(self):
        if not self.frames: return 0.0
        return sum(f.detected for f in self.frames) / len(self.frames)

    @property
    def mean_add_cm(self):
        valid = [f.add_cm for f in self.frames if f.detected and np.isfinite(f.add_cm)]
        return float(np.mean(valid)) if valid else float('inf')

    @property
    def median_add_cm(self):
        valid = [f.add_cm for f in self.frames if f.detected and np.isfinite(f.add_cm)]
        return float(np.median(valid)) if valid else float('inf')

    def auc(self, threshold_cm=10.0):
        valid = [f.add_cm for f in self.frames if f.detected and np.isfinite(f.add_cm)]
        if not valid: return 0.0
        return sum(v < threshold_cm for v in valid) / len(valid)

    @property
    def add_values(self):
        return [f.add_cm for f in self.frames if f.detected and np.isfinite(f.add_cm)]


def compute_add(model_pts: np.ndarray,
                pose_gt:   np.ndarray,
                pose_est:  np.ndarray) -> float:
    """ADD score in centimetres."""
    pts_gt  = (pose_gt[:3,:3]  @ model_pts.T).T + pose_gt[:3,3]
    pts_est = (pose_est[:3,:3] @ model_pts.T).T + pose_est[:3,3]
    return float(np.mean(np.linalg.norm(pts_gt - pts_est, axis=1))) * 100.0


def sample_mesh_points(mesh: trimesh.Trimesh, n: int = 1000) -> np.ndarray:
    pts, _ = trimesh.sample.sample_surface(mesh, n)
    return pts.astype(np.float64)


print('✅ Data structures defined')

---
## Cell 4 — Dataset Loader

In [ ]:
def load_rgbd_sequence(dataset_root, category, instance, seq_id, n_frames=10):
    """
    Load n_frames evenly spaced from one RGB-D sequence.
    Returns list of (rgb H×W×3, depth H×W float32 metres, pose_gt 4×4, frame_index).
    """
    seq_dir = Path(dataset_root) / category / instance / f'{instance}_{seq_id}'
    if not seq_dir.exists():
        raise FileNotFoundError(f'Sequence not found: {seq_dir}')

    colour_files = sorted([
        f for f in seq_dir.glob('*.png')
        if '_depth' not in f.name and '_loc' not in f.name
    ])
    if not colour_files:
        raise FileNotFoundError(f'No colour frames in {seq_dir}')

    indices = np.linspace(0, len(colour_files) - 1, n_frames, dtype=int)
    frames  = []

    for idx in indices:
        cp  = colour_files[idx]
        dp  = seq_dir / f'{cp.stem}_depth.png'
        pp  = seq_dir / f'{cp.stem}.txt'

        rgb   = cv2.cvtColor(cv2.imread(str(cp)), cv2.COLOR_BGR2RGB)
        depth = cv2.imread(str(dp), cv2.IMREAD_ANYDEPTH).astype(np.float32) / DEPTH_SCALE
        pose  = np.loadtxt(str(pp)).reshape(4,4) if pp.exists() else np.eye(4)

        frames.append((rgb, depth, pose, int(idx)))

    return frames


print('✅ Dataset loader defined')

---
## Cell 5 — Initialise Models

In [ ]:
print('Loading YOLOE...')
yoloe_model = YOLOE(YOLOE_WEIGHTS)
yoloe_model.to('cuda')
print('  ✅ YOLOE ready')

print('Initialising CUDA rasterisation context...')
glctx = dr.RasterizeCudaContext()
print('  ✅ glctx ready')

# Any6D estimator cache — one per object (avoids reinitialisation overhead)
estimator_cache = {}

---
## Cell 6 — Detection Helper (YOLOE)

In [ ]:
def detect_with_yoloe(anchor_rgb, scene_rgb):
    """
    Visual-prompt detection: anchor_rgb acts as the reference.
    Returns (bbox_xyxy [4], mask_bool H×W) or (None, None).
    """
    with tempfile.TemporaryDirectory() as tmp:
        a_path = os.path.join(tmp, 'anchor.jpg')
        s_path = os.path.join(tmp, 'scene.jpg')
        Image.fromarray(anchor_rgb).save(a_path)
        Image.fromarray(scene_rgb).save(s_path)

        h, w = anchor_rgb.shape[:2]
        vp   = {'cls': [0], 'bboxes': np.array([[0, 0, w, h]])}

        try:
            results = yoloe_model.predict(
                s_path,
                refer_image=a_path,
                visual_prompts=vp,
                predictor=YOLOEVPSegPredictor,
                verbose=False
            )
        except Exception:
            return None, None

    result = results[0]
    if len(result.boxes) == 0:
        return None, None

    best  = result.boxes.conf.argmax().item()
    bbox  = result.boxes.xyxy[best].cpu().numpy().astype(int)

    if result.masks is not None:
        m = result.masks.data[best].cpu().numpy()
        m = cv2.resize(m, (scene_rgb.shape[1], scene_rgb.shape[0]))
        mask = m > 0.5
    else:
        mask = np.zeros(scene_rgb.shape[:2], dtype=bool)
        x1,y1,x2,y2 = bbox
        mask[y1:y2, x1:x2] = True

    return bbox, mask


print('✅ YOLOE helper defined')

---
## Cell 7 — Run Evaluation Loop

In [ ]:
all_results: List[ObjectResult] = []

for category, instance, seq_id in EVAL_OBJECTS:

    mesh_path = os.path.join(MESH_ROOT, f'{instance}.obj')
    if not os.path.exists(mesh_path):
        print(f'[SKIP] mesh not found: {mesh_path}')
        continue

    # ── Initialise estimator for this object ──────────────────
    mesh = trimesh.load(mesh_path)
    if instance not in estimator_cache:
        dbg = str(SAVE_DIR / instance)
        os.makedirs(dbg, exist_ok=True)
        estimator_cache[instance] = Any6D(
            symmetry_tfs=None, mesh=mesh, debug_dir=dbg, debug=0
        )
    estimator = estimator_cache[instance]
    model_pts = sample_mesh_points(mesh)

    # ── Load frames ───────────────────────────────────────────
    frames = load_rgbd_sequence(DATASET_ROOT, category, instance, seq_id, N_FRAMES)
    anchor_rgb = frames[0][0]

    obj_result = ObjectResult(name=instance)
    print(f'\n▶  {instance:<30}  ({len(frames)} frames)')

    for rgb, depth, gt_pose, frame_idx in frames:

        # Detection
        bbox, mask = detect_with_yoloe(anchor_rgb, rgb)

        if mask is None or mask.sum() < 100:
            print(f'   frame {frame_idx:04d}  │ not detected')
            obj_result.frames.append(FrameResult(
                object_name=instance, frame_id=frame_idx,
                add_cm=float('inf'), detected=False,
                pred_pose=np.eye(4), gt_pose=gt_pose
            ))
            continue

        # Pose estimation
        try:
            pred_pose = estimator.register_any6d(
                K=K_RGBD, rgb=rgb, depth=depth,
                ob_mask=mask, iteration=5,
                name=f'{instance}_{frame_idx:04d}'
            )
            add = compute_add(model_pts, gt_pose, pred_pose)
            print(f'   frame {frame_idx:04d}  │ ADD = {add:6.2f} cm  {"✓" if add < ADD_THRESHOLD else "✗"}')
            obj_result.frames.append(FrameResult(
                object_name=instance, frame_id=frame_idx,
                add_cm=add, detected=True,
                pred_pose=pred_pose, gt_pose=gt_pose
            ))
        except Exception as e:
            print(f'   frame {frame_idx:04d}  │ pose error: {e}')
            obj_result.frames.append(FrameResult(
                object_name=instance, frame_id=frame_idx,
                add_cm=float('inf'), detected=True,
                pred_pose=np.eye(4), gt_pose=gt_pose
            ))

    print(f'   → det {obj_result.detection_rate*100:.0f}%  │  mean ADD {obj_result.mean_add_cm:.2f} cm  │  AUC@10 {obj_result.auc(ADD_THRESHOLD)*100:.0f}%')
    all_results.append(obj_result)

# ── Save raw results ──────────────────────────────────────────
raw = [
    {
        'object':         r.name,
        'detection_rate': round(r.detection_rate, 4),
        'mean_add_cm':    round(r.mean_add_cm, 4) if np.isfinite(r.mean_add_cm) else None,
        'median_add_cm':  round(r.median_add_cm, 4) if np.isfinite(r.median_add_cm) else None,
        'auc_10cm':       round(r.auc(ADD_THRESHOLD), 4),
        'add_values':     [round(v,4) for v in r.add_values],
    }
    for r in all_results
]
with open(SAVE_DIR / 'results.json', 'w') as f:
    json.dump(raw, f, indent=2)

print('\n✅ Evaluation complete — results.json saved')

---
## Cell 8 — Summary Table

In [ ]:
print(f'\n{"═"*68}')
print(f'{"Object":<28} {"Det%":>6}  {"Mean ADD":>9}  {"Median ADD":>11}  {"AUC@10":>7}')
print(f'{"─"*68}')

for r in all_results:
    add_str = f'{r.mean_add_cm:.2f} cm' if np.isfinite(r.mean_add_cm) else '      —'
    med_str = f'{r.median_add_cm:.2f} cm' if np.isfinite(r.median_add_cm) else '      —'
    print(f'{r.name:<28} {r.detection_rate*100:>5.0f}%  {add_str:>9}  {med_str:>11}  {r.auc(ADD_THRESHOLD)*100:>6.0f}%')

print(f'{"─"*68}')
all_det  = np.mean([r.detection_rate for r in all_results])
all_add  = np.mean([r.mean_add_cm for r in all_results if np.isfinite(r.mean_add_cm)])
all_auc  = np.mean([r.auc(ADD_THRESHOLD) for r in all_results])
print(f'{"MEAN (10 objects)":<28} {all_det*100:>5.0f}%  {all_add:>7.2f} cm              {all_auc*100:>6.0f}%')
print(f'{"═"*68}')

---
## Cell 9 — Plot 1 · ADD Score per Object (horizontal bar)

In [ ]:
names     = [r.name.replace('_', ' ') for r in all_results]
means     = [r.mean_add_cm   if np.isfinite(r.mean_add_cm)   else 0 for r in all_results]
medians   = [r.median_add_cm if np.isfinite(r.median_add_cm) else 0 for r in all_results]
aucs      = [r.auc(ADD_THRESHOLD) * 100 for r in all_results]
det_rates = [r.detection_rate * 100 for r in all_results]

y = np.arange(len(names))
h = 0.35

fig, ax = plt.subplots(figsize=(9, 5.5))

bars_mean   = ax.barh(y + h/2, means,   h, label='Mean ADD',   color=C_BLUE,   alpha=0.85)
bars_median = ax.barh(y - h/2, medians, h, label='Median ADD', color=C_GREEN,  alpha=0.85)

# Threshold line
ax.axvline(ADD_THRESHOLD, color=C_RED, lw=1.4, ls='--', label=f'{ADD_THRESHOLD:.0f} cm threshold')

# Value labels
for bar in bars_mean:
    w = bar.get_width()
    if w > 0:
        ax.text(w + 0.15, bar.get_y() + bar.get_height()/2,
                f'{w:.1f}', va='center', ha='left', fontsize=8, color=C_DARK)

ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('ADD score (cm)', fontsize=11)
ax.set_title('Pose Accuracy — ADD Score per Object', fontsize=12, fontweight='bold', pad=12)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, max(means + [ADD_THRESHOLD]) * 1.25)

# Subtle alternating row background
for i in range(0, len(names), 2):
    ax.axhspan(i - 0.5, i + 0.5, color='#F4F6F7', zorder=0)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'plot1_add_per_object.pdf', bbox_inches='tight')
plt.savefig(SAVE_DIR / 'plot1_add_per_object.png', bbox_inches='tight', dpi=200)
plt.show()

---
## Cell 10 — Plot 2 · Detection Rate + AUC@10 (grouped)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

bars_det = ax.barh(y + h/2, det_rates, h, label='Detection rate (%)', color=C_ORANGE, alpha=0.85)
bars_auc = ax.barh(y - h/2, aucs,      h, label='AUC@10 cm (%)',      color=C_BLUE,  alpha=0.85)

for bar in bars_det:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height()/2,
            f'{w:.0f}%', va='center', ha='left', fontsize=8, color=C_DARK)

for bar in bars_auc:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height()/2,
            f'{w:.0f}%', va='center', ha='left', fontsize=8, color=C_DARK)

ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Percentage (%)', fontsize=11)
ax.set_xlim(0, 120)
ax.set_title('Detection Rate & Pose Success (AUC@10 cm)', fontsize=12, fontweight='bold', pad=12)
ax.legend(loc='lower right', fontsize=9)

for i in range(0, len(names), 2):
    ax.axhspan(i - 0.5, i + 0.5, color='#F4F6F7', zorder=0)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'plot2_detection_auc.pdf', bbox_inches='tight')
plt.savefig(SAVE_DIR / 'plot2_detection_auc.png', bbox_inches='tight', dpi=200)
plt.show()

---
## Cell 11 — Plot 3 · ADD Distribution (violin / box)

In [ ]:
# Collect per-object ADD values (only valid/detected frames)
add_data = [r.add_values for r in all_results]
add_data_clean = [d if d else [0.0] for d in add_data]  # avoid empty lists

fig, ax = plt.subplots(figsize=(12, 4.5))

# Violin
parts = ax.violinplot(
    add_data_clean,
    positions=np.arange(len(names)),
    widths=0.6,
    showmedians=True,
    showextrema=False
)

for pc in parts['bodies']:
    pc.set_facecolor(C_BLUE)
    pc.set_alpha(0.45)
    pc.set_edgecolor(C_DARK)
    pc.set_linewidth(0.8)

parts['cmedians'].set_color(C_DARK)
parts['cmedians'].set_linewidth(1.6)

# Scatter individual points
for i, vals in enumerate(add_data_clean):
    jitter = np.random.uniform(-0.08, 0.08, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals,
               s=18, color=C_BLUE, alpha=0.65, zorder=3, lw=0)

# Threshold line
ax.axhline(ADD_THRESHOLD, color=C_RED, lw=1.4, ls='--',
           label=f'{ADD_THRESHOLD:.0f} cm threshold (AUC criterion)')

ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=28, ha='right', fontsize=9)
ax.set_ylabel('ADD score (cm)', fontsize=11)
ax.set_title('ADD Score Distribution per Object', fontsize=12, fontweight='bold', pad=12)
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'plot3_add_distribution.pdf', bbox_inches='tight')
plt.savefig(SAVE_DIR / 'plot3_add_distribution.png', bbox_inches='tight', dpi=200)
plt.show()

---
## Cell 12 — Plot 4 · AUC Curve (ADD threshold sweep)

In [ ]:
thresholds = np.linspace(0, 20, 200)

fig, ax = plt.subplots(figsize=(8, 5))

cmap   = plt.cm.get_cmap('tab10', len(all_results))
curves = []

for i, r in enumerate(all_results):
    vals = r.add_values
    if not vals:
        continue
    auc_curve = [sum(v < t for v in vals) / len(vals) * 100 for t in thresholds]
    ax.plot(thresholds, auc_curve, lw=1.4, alpha=0.8,
            color=cmap(i), label=r.name.replace('_', ' '))
    curves.append(auc_curve)

# Mean curve
if curves:
    mean_curve = np.mean(curves, axis=0)
    ax.plot(thresholds, mean_curve, lw=2.5, color=C_DARK,
            ls='--', label='Mean (all objects)', zorder=10)

ax.axvline(ADD_THRESHOLD, color=C_RED, lw=1.2, ls=':', alpha=0.8)
ax.text(ADD_THRESHOLD + 0.2, 5, f'{ADD_THRESHOLD:.0f} cm', color=C_RED, fontsize=8)

ax.set_xlabel('ADD threshold (cm)', fontsize=11)
ax.set_ylabel('Frames below threshold (%)', fontsize=11)
ax.set_title('AUC Curve — ADD Score vs Threshold', fontsize=12, fontweight='bold', pad=12)
ax.set_xlim(0, 20)
ax.set_ylim(0, 105)
ax.legend(fontsize=7.5, ncol=2, loc='lower right')
ax.fill_between(thresholds,
                np.min(curves, axis=0) if curves else thresholds*0,
                np.max(curves, axis=0) if curves else thresholds*0,
                alpha=0.07, color=C_BLUE)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'plot4_auc_curve.pdf', bbox_inches='tight')
plt.savefig(SAVE_DIR / 'plot4_auc_curve.png', bbox_inches='tight', dpi=200)
plt.show()

---
## Cell 13 — Plot 5 · Scatter — Detection Confidence vs ADD

In [ ]:
# Requires that FrameResult stores YOLOE confidence.
# If confidence was not recorded, this cell produces a detection-rate vs mean-ADD scatter.

fig, ax = plt.subplots(figsize=(7, 5))

cmap2 = plt.cm.get_cmap('tab10', len(all_results))

for i, r in enumerate(all_results):
    if not r.add_values:
        continue
    ax.scatter(
        r.detection_rate * 100,
        r.mean_add_cm,
        s=100,
        color=cmap2(i),
        zorder=4,
        label=r.name.replace('_', ' '),
        edgecolors='white',
        linewidths=0.8
    )
    ax.annotate(
        r.name.replace('_', ' '),
        xy=(r.detection_rate * 100, r.mean_add_cm),
        xytext=(4, 2), textcoords='offset points',
        fontsize=8, color=C_DARK
    )

ax.axhline(ADD_THRESHOLD, color=C_RED, lw=1.2, ls='--', alpha=0.7,
           label=f'{ADD_THRESHOLD:.0f} cm threshold')

ax.set_xlabel('Detection rate (%)', fontsize=11)
ax.set_ylabel('Mean ADD (cm)', fontsize=11)
ax.set_title('Detection Rate vs Pose Accuracy', fontsize=12, fontweight='bold', pad=12)
ax.set_xlim(0, 110)
ax.set_ylim(bottom=0)
ax.legend(fontsize=7.5, ncol=2, loc='upper right')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'plot5_det_vs_add_scatter.pdf', bbox_inches='tight')
plt.savefig(SAVE_DIR / 'plot5_det_vs_add_scatter.png', bbox_inches='tight', dpi=200)
plt.show()

---
## Cell 14 — Plot 6 · Dashboard (thesis-ready 2×3 summary figure)

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

ax1 = fig.add_subplot(gs[0, 0])   # ADD bar
ax2 = fig.add_subplot(gs[0, 1])   # AUC bar
ax3 = fig.add_subplot(gs[0, 2])   # Detection bar
ax4 = fig.add_subplot(gs[1, 0])   # Violin
ax5 = fig.add_subplot(gs[1, 1])   # AUC curve
ax6 = fig.add_subplot(gs[1, 2])   # Scatter

short_names = [n.split(' ')[0] for n in names]   # first word only for tight axes

# ── (a) Mean ADD ──────────────────────────────────────────────
colors_add = [C_GREEN if m < ADD_THRESHOLD else C_RED for m in means]
ax1.barh(short_names, means, color=colors_add, alpha=0.85)
ax1.axvline(ADD_THRESHOLD, color=C_RED, lw=1.2, ls='--')
ax1.set_xlabel('Mean ADD (cm)', fontsize=9)
ax1.set_title('(a) Pose Accuracy', fontsize=10, fontweight='bold')
ax1.tick_params(labelsize=8)

# ── (b) AUC@10 ───────────────────────────────────────────────
ax2.barh(short_names, aucs, color=C_BLUE, alpha=0.85)
ax2.set_xlabel('AUC@10 cm (%)', fontsize=9)
ax2.set_title('(b) AUC@10 cm', fontsize=10, fontweight='bold')
ax2.set_xlim(0, 110)
ax2.tick_params(labelsize=8)

# ── (c) Detection rate ────────────────────────────────────────
ax3.barh(short_names, det_rates, color=C_ORANGE, alpha=0.85)
ax3.set_xlabel('Detection rate (%)', fontsize=9)
ax3.set_title('(c) YOLOE Detection', fontsize=10, fontweight='bold')
ax3.set_xlim(0, 110)
ax3.tick_params(labelsize=8)

# ── (d) Violin ────────────────────────────────────────────────
if any(add_data_clean):
    parts2 = ax4.violinplot(add_data_clean, positions=np.arange(len(names)),
                            widths=0.55, showmedians=True, showextrema=False)
    for pc in parts2['bodies']:
        pc.set_facecolor(C_BLUE); pc.set_alpha(0.4); pc.set_edgecolor(C_DARK); pc.set_linewidth(0.7)
    parts2['cmedians'].set_color(C_DARK); parts2['cmedians'].set_linewidth(1.4)
    for i, vals in enumerate(add_data_clean):
        ax4.scatter(np.full(len(vals), i) + np.random.uniform(-0.07,0.07,len(vals)),
                    vals, s=12, color=C_BLUE, alpha=0.6, zorder=3, lw=0)
ax4.axhline(ADD_THRESHOLD, color=C_RED, lw=1.2, ls='--')
ax4.set_xticks(np.arange(len(short_names)))
ax4.set_xticklabels(short_names, rotation=30, ha='right', fontsize=7)
ax4.set_ylabel('ADD (cm)', fontsize=9)
ax4.set_title('(d) ADD Distribution', fontsize=10, fontweight='bold')
ax4.set_ylim(bottom=0)

# ── (e) AUC curve ─────────────────────────────────────────────
cmap3 = plt.cm.get_cmap('tab10', len(all_results))
curves2 = []
for i, r in enumerate(all_results):
    vals = r.add_values
    if not vals: continue
    c = [sum(v < t for v in vals)/len(vals)*100 for t in thresholds]
    ax5.plot(thresholds, c, lw=1.0, alpha=0.6, color=cmap3(i))
    curves2.append(c)
if curves2:
    ax5.plot(thresholds, np.mean(curves2,axis=0), lw=2.2, color=C_DARK, ls='--', label='Mean')
ax5.axvline(ADD_THRESHOLD, color=C_RED, lw=1.0, ls=':')
ax5.set_xlabel('Threshold (cm)', fontsize=9)
ax5.set_ylabel('Frames below (%)', fontsize=9)
ax5.set_title('(e) AUC Curve', fontsize=10, fontweight='bold')
ax5.set_xlim(0,20); ax5.set_ylim(0,105)
ax5.legend(fontsize=8)
ax5.tick_params(labelsize=8)

# ── (f) Scatter det vs ADD ────────────────────────────────────
for i, r in enumerate(all_results):
    if not r.add_values: continue
    ax6.scatter(r.detection_rate*100, r.mean_add_cm,
                s=80, color=cmap3(i), zorder=4, edgecolors='white', lw=0.6)
    ax6.annotate(r.name.replace('_',' '),
                 (r.detection_rate*100, r.mean_add_cm),
                 xytext=(3,2), textcoords='offset points', fontsize=7, color=C_DARK)
ax6.axhline(ADD_THRESHOLD, color=C_RED, lw=1.0, ls='--', alpha=0.7)
ax6.set_xlabel('Detection rate (%)', fontsize=9)
ax6.set_ylabel('Mean ADD (cm)', fontsize=9)
ax6.set_title('(f) Detection vs Accuracy', fontsize=10, fontweight='bold')
ax6.set_xlim(0,110); ax6.set_ylim(bottom=0)
ax6.tick_params(labelsize=8)

fig.suptitle(
    'YOLOE → Any6D · Evaluation on the RGB-D Object Dataset (10 objects)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig(SAVE_DIR / 'plot6_dashboard.pdf', bbox_inches='tight')
plt.savefig(SAVE_DIR / 'plot6_dashboard.png', bbox_inches='tight', dpi=200)
plt.show()
print(f'\nDashboard saved → {SAVE_DIR / "plot6_dashboard.pdf"}')

---
## Cell 15 — Qualitative Visualisation (detection overlay per object)

In [ ]:
def make_overlay(rgb, mask, bbox, add_cm=None, detected=True):
    """Overlay mask + bbox + ADD label on an RGB frame."""
    out = rgb.copy()
    if detected and mask is not None:
        tint = np.zeros_like(out)
        tint[mask] = [0, 180, 100]
        out = cv2.addWeighted(out, 0.75, tint, 0.25, 0)
        if bbox is not None:
            x1,y1,x2,y2 = bbox
            cv2.rectangle(out, (x1,y1), (x2,y2), (0,180,100), 2)
    label = f'ADD {add_cm:.1f} cm' if (add_cm is not None and np.isfinite(add_cm)) else 'not detected'
    colour = (40,200,100) if (add_cm and np.isfinite(add_cm) and add_cm < ADD_THRESHOLD) else (200,60,60)
    cv2.putText(out, label, (8, out.shape[0]-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2, cv2.LINE_AA)
    return out


n_objs = len(all_results)
fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
axes = axes.flatten()

for ax_i, r in enumerate(all_results):
    ax = axes[ax_i]

    # Load best frame (lowest ADD, or first detected)
    valid_frames = [f for f in r.frames if f.detected and np.isfinite(f.add_cm)]
    if not valid_frames:
        ax.text(0.5, 0.5, 'No valid\nframe', ha='center', va='center',
                transform=ax.transAxes, fontsize=9, color='grey')
        ax.set_title(r.name.replace('_',' '), fontsize=8, fontweight='bold')
        ax.axis('off')
        continue

    best_frame = min(valid_frames, key=lambda f: f.add_cm)
    cat = next(c for c, inst, _ in EVAL_OBJECTS if inst == r.name)
    seq = next(s for _, inst, s in EVAL_OBJECTS if inst == r.name)

    seq_dir = Path(DATASET_ROOT) / cat / r.name / f'{r.name}_{seq}'
    colour_files = sorted([f for f in seq_dir.glob('*.png')
                           if '_depth' not in f.name and '_loc' not in f.name])
    rgb_frame = cv2.cvtColor(
        cv2.imread(str(colour_files[best_frame.frame_id])), cv2.COLOR_BGR2RGB
    )

    anchor_rgb = cv2.cvtColor(cv2.imread(str(colour_files[0])), cv2.COLOR_BGR2RGB)
    bbox_q, mask_q = detect_with_yoloe(anchor_rgb, rgb_frame)

    overlay = make_overlay(rgb_frame, mask_q, bbox_q, best_frame.add_cm)
    ax.imshow(overlay)

    status = '✓' if best_frame.add_cm < ADD_THRESHOLD else '✗'
    ax.set_title(f'{r.name.replace("_"," ")}\n{status} ADD={best_frame.add_cm:.1f} cm',
                 fontsize=8, fontweight='bold',
                 color=C_GREEN if best_frame.add_cm < ADD_THRESHOLD else C_RED)
    ax.axis('off')

for ax in axes[n_objs:]:
    ax.axis('off')

fig.suptitle(
    'Qualitative Results — Best Frame per Object (green mask = detected region)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'plot7_qualitative.pdf', bbox_inches='tight')
plt.savefig(SAVE_DIR / 'plot7_qualitative.png', bbox_inches='tight', dpi=200)
plt.show()

---
## Cell 16 — Export Summary (LaTeX table)

In [ ]:
latex_rows = []
for r in all_results:
    add_str = f'{r.mean_add_cm:.2f}' if np.isfinite(r.mean_add_cm) else '—'
    med_str = f'{r.median_add_cm:.2f}' if np.isfinite(r.median_add_cm) else '—'
    latex_rows.append(
        f'  {r.name.replace("_"," "):<28} & '
        f'{r.detection_rate*100:.0f}\\% & '
        f'{add_str} & '
        f'{med_str} & '
        f'{r.auc(ADD_THRESHOLD)*100:.0f}\\% \\\\'
    )

latex = r"""\begin{table}[ht]
\centering
\caption{Quantitative results on the RGB-D Object Dataset. ADD is reported in centimetres.
         AUC@10 cm denotes the fraction of frames with ADD below 10\,cm.}
\label{tab:eval_rgbd}
\begin{tabular}{lrrrr}
\toprule
Object & Detection & Mean ADD (cm) & Median ADD (cm) & AUC@10\,cm \\\\
\midrule
""" + '\n'.join(latex_rows) + r"""
\midrule
  \textbf{Mean} & """

all_det_mean = np.mean([r.detection_rate for r in all_results])
all_add_mean = np.mean([r.mean_add_cm for r in all_results if np.isfinite(r.mean_add_cm)])
all_auc_mean = np.mean([r.auc(ADD_THRESHOLD) for r in all_results])

latex += (
    f'\\textbf{{{all_det_mean*100:.0f}\\%}} & '
    f'\\textbf{{{all_add_mean:.2f}}} & — & '
    f'\\textbf{{{all_auc_mean*100:.0f}\\%}} \\\\\n'
    r'\bottomrule' + '\n'
    r'\end{tabular}' + '\n'
    r'\end{table}'
)

print(latex)

with open(SAVE_DIR / 'table_results.tex', 'w') as f:
    f.write(latex)

print(f'\nLaTeX table saved → {SAVE_DIR / "table_results.tex"}')

---
## Cell 17 — Output File Index

In [ ]:
print('Files saved to:', SAVE_DIR)
for f in sorted(SAVE_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size:>7.1f} kB')